<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Custom embedddings con Gensim



### Objetivo
El objetivo es utilizar documentos / corpus para crear embeddings de palabras basado en ese contexto. Se utilizará canciones de bandas para generar los embeddings, es decir, que los vectores tendrán la forma en función de como esa banda haya utilizado las palabras en sus canciones.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import multiprocessing
try:
  from gensim.models import Word2Vec
except:
  !pip install gensim
  from gensim.models import Word2Vec

### Datos
Utilizaremos como dataset canciones de bandas de habla inglesa.

In [2]:
# Descargar la carpeta de dataset
import os
import platform
if os.access('./songs_dataset', os.F_OK) is False:
    if os.access('songs_dataset.zip', os.F_OK) is False:
        if platform.system() == 'Windows':
            !curl https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip -o songs_dataset.zip
        else:
            !wget songs_dataset.zip https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
    !unzip -q songs_dataset.zip
else:
    print("El dataset ya se encuentra descargado")

El dataset ya se encuentra descargado


In [3]:
# Posibles bandas
os.listdir("./songs_dataset/")

['adele.txt',
 'al-green.txt',
 'alicia-keys.txt',
 'amy-winehouse.txt',
 'beatles.txt',
 'bieber.txt',
 'bjork.txt',
 'blink-182.txt',
 'bob-dylan.txt',
 'bob-marley.txt',
 'britney-spears.txt',
 'bruce-springsteen.txt',
 'bruno-mars.txt',
 'cake.txt',
 'dickinson.txt',
 'disney.txt',
 'dj-khaled.txt',
 'dolly-parton.txt',
 'dr-seuss.txt',
 'drake.txt',
 'eminem.txt',
 'janisjoplin.txt',
 'jimi-hendrix.txt',
 'johnny-cash.txt',
 'joni-mitchell.txt',
 'kanye-west.txt',
 'kanye.txt',
 'Kanye_West.txt',
 'lady-gaga.txt',
 'leonard-cohen.txt',
 'lil-wayne.txt',
 'Lil_Wayne.txt',
 'lin-manuel-miranda.txt',
 'lorde.txt',
 'ludacris.txt',
 'michael-jackson.txt',
 'missy-elliott.txt',
 'nickelback.txt',
 'nicki-minaj.txt',
 'nirvana.txt',
 'notorious-big.txt',
 'notorious_big.txt',
 'nursery_rhymes.txt',
 'patti-smith.txt',
 'paul-simon.txt',
 'prince.txt',
 'r-kelly.txt',
 'radiohead.txt',
 'rihanna.txt']

In [4]:
# Armar el dataset utilizando salto de línea para separar las oraciones/docs
df = pd.read_csv('songs_dataset/beatles.txt', sep='/n', header=None)
df.head()

/tmp/ipykernel_49798/3849064916.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv('songs_dataset/beatles.txt', sep='/n', header=None)


,0
0,"Yesterday, all my troubles seemed so far away"
1,Now it looks as though they're here to stay
2,"Oh, I believe in yesterday Suddenly, I'm not h..."
3,There's a shadow hanging over me.
4,"Oh, yesterday came suddenly Why she had to go ..."


In [5]:
print("Cantidad de documentos:", df.shape[0])

Cantidad de documentos: 1846


### 1 - Preprocesamiento

In [6]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence

sentence_tokens = []
# Recorrer todas las filas y transformar las oraciones
# en una secuencia de palabras (esto podría realizarse con NLTK o spaCy también)
for _, row in df[:None].iterrows():
    sentence_tokens.append(text_to_word_sequence(row[0]))

I0000 00:00:1786900134.722237   49798 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786900134.800152   49798 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786900135.703286   49798 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [7]:
# Demos un vistazo
sentence_tokens[:2]

[['yesterday', 'all', 'my', 'troubles', 'seemed', 'so', 'far', 'away'],
 ['now', 'it', 'looks', 'as', 'though', "they're", 'here', 'to', 'stay']]

### 2 - Crear los vectores (word2vec)

In [8]:
from gensim.models.callbacks import CallbackAny2Vec
# Durante el entrenamiento gensim por defecto no informa el "loss" en cada época
# Sobrecargamos el callback para poder tener esta información
class callback(CallbackAny2Vec):
    """
    Callback to print loss after each epoch
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [9]:
# Crearmos el modelo generador de vectores
# En este caso utilizaremos la estructura modelo Skipgram
w2v_model = Word2Vec(min_count=5,    # frecuencia mínima de palabra para incluirla en el vocabulario
                     window=2,       # cant de palabras antes y desp de la predicha
                     vector_size=300,       # dimensionalidad de los vectores
                     negative=20,    # cantidad de negative samples... 0 es no se usa
                     workers=1,      # si tienen más cores pueden cambiar este valor
                     sg=1)           # modelo 0:CBOW  1:skipgram

In [10]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

In [11]:
# Cantidad de filas/docs encontradas en el corpus
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)

Cantidad de docs en el corpus: 1846


In [12]:
# Cantidad de words encontradas en el corpus
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))

Cantidad de words distintas en el corpus: 445


### 3 - Entrenar embeddings

In [13]:
# Entrenamos el modelo generador de vectores
# Utilizamos nuestro callback
w2v_model.train(sentence_tokens,
                 total_examples=w2v_model.corpus_count,
                 epochs=20,
                 compute_loss = True,
                 callbacks=[callback()]
                 )

Loss after epoch 0: 113045.25
Loss after epoch 1: 65966.59375
Loss after epoch 2: 65934.984375
Loss after epoch 3: 65718.390625
Loss after epoch 4: 63875.09375
Loss after epoch 5: 64160.65625
Loss after epoch 6: 64080.21875
Loss after epoch 7: 64814.875
Loss after epoch 8: 62632.75
Loss after epoch 9: 60452.875
Loss after epoch 10: 59839.875
Loss after epoch 11: 58884.375
Loss after epoch 12: 57715.75
Loss after epoch 13: 56494.3125
Loss after epoch 14: 55817.5
Loss after epoch 15: 55842.9375
Loss after epoch 16: 51722.4375
Loss after epoch 17: 49858.0
Loss after epoch 18: 49592.25
Loss after epoch 19: 48960.125


(156986, 287740)

### 4 - Ensayar

In [14]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["darling"], topn=10)

[('pretty', 0.8954247832298279),
 ('sleep', 0.8665655851364136),
 ('help', 0.8439376354217529),
 ('cry', 0.8351269960403442),
 ('not', 0.8309612274169922),
 ('try', 0.8276943564414978),
 ('peace', 0.8144856691360474),
 ('little', 0.8140572309494019),
 ('twist', 0.8123919367790222),
 ('seems', 0.8079564571380615)]

In [15]:
# Palabras que MENOS se relacionan con...:
w2v_model.wv.most_similar(negative=["love"], topn=10)

[('shake', -0.22873197495937347),
 ('four', -0.2330218255519867),
 ('five', -0.23746445775032043),
 ('six', -0.23784494400024414),
 ('bang', -0.24832050502300262),
 ('our', -0.25539135932922363),
 ('day', -0.2689811885356903),
 ('going', -0.2692062556743622),
 ('here', -0.26990723609924316),
 ('three', -0.2838989198207855)]

In [16]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["four"], topn=10)

[('five', 0.9813723564147949),
 ('three', 0.9745770692825317),
 ('six', 0.9710808992385864),
 ('seven', 0.9584357738494873),
 ('two', 0.9517216682434082),
 ('sixty', 0.8990395665168762),
 ('one', 0.7951181530952454),
 ('crying', 0.7946289777755737),
 ('us', 0.7740051746368408),
 ("i'm", 0.7508383393287659)]

In [17]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["money"], topn=5)

[("can't", 0.9434017539024353),
 ('buy', 0.9396998882293701),
 ('much', 0.9033146500587463),
 ('just', 0.8509082198143005),
 ('hide', 0.835538387298584)]

In [18]:
# Ensayar con una palabra que no está en el vocabulario:
try:
    w2v_model.wv.most_similar(negative=["diedaa"])
except KeyError as e:
    print(f"KeyError: {e} — la palabra no está en el vocabulario")


KeyError: "Key 'diedaa' not present in vocabulary" — la palabra no está en el vocabulario


In [19]:
# el método `get_vector` permite obtener los vectores:
vector_love = w2v_model.wv.get_vector("love")
print(vector_love)

[ 0.06138203  0.05881222 -0.06370417  0.02444947 -0.20152196 -0.18612292
 -0.15284595  0.4548753  -0.04217871  0.03536078  0.13657516 -0.18520005
 -0.1812647   0.22149836 -0.3038084  -0.23970386  0.07094695 -0.05679139
 -0.05166207 -0.23843557 -0.08530281  0.19564727 -0.07678778  0.03797247
  0.07517307 -0.04826551  0.07379535  0.10396848  0.00738022 -0.22764729
 -0.0456724   0.12937619  0.27785638  0.19387618 -0.13509148  0.20857106
  0.40917322 -0.00387122 -0.1063128  -0.09056759  0.02400028 -0.0800491
  0.13400665  0.08833536 -0.01894405  0.08592905 -0.15905626  0.10259357
  0.14459287 -0.12092585 -0.27919102 -0.04061577  0.11382084  0.31365854
 -0.07409792  0.13976744  0.22791271  0.13209458 -0.01811365  0.09772275
  0.09249583 -0.14871688 -0.16348091 -0.13203284 -0.09834065  0.02714608
  0.16531324  0.26051944 -0.0325964  -0.02894551  0.11621328 -0.06974234
  0.09563565 -0.15276384  0.22071053  0.15996666  0.1589048  -0.04711676
 -0.12555045 -0.03993924 -0.10795183  0.01878959  0.

In [20]:
# el método `most_similar` también permite comparar a partir de vectores
w2v_model.wv.most_similar(vector_love)

[('love', 0.9999999403953552),
 ('babe', 0.9085132479667664),
 ('someone', 0.8886148929595947),
 ('need', 0.8827974200248718),
 ('nothing', 0.8740269541740417),
 ("didn't", 0.8638361096382141),
 ("there's", 0.8526672720909119),
 ('you', 0.8456704616546631),
 ('feed', 0.8445017337799072),
 ('somebody', 0.8362804651260376)]

In [21]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["love"], topn=10)

[('babe', 0.9085132479667664),
 ('someone', 0.8886148929595947),
 ('need', 0.8827974200248718),
 ('nothing', 0.8740269541740417),
 ("didn't", 0.8638360500335693),
 ("there's", 0.8526672720909119),
 ('you', 0.8456703424453735),
 ('feed', 0.8445016741752625),
 ('somebody', 0.8362804651260376),
 ('buy', 0.8351694941520691)]

### 5 - Visualizar agrupación de vectores

In [22]:
from sklearn.decomposition import IncrementalPCA
from sklearn.manifold import TSNE
import numpy as np

def reduce_dimensions(model, num_dimensions = 2 ):

    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)

    tsne = TSNE(n_components=num_dimensions, random_state=0)
    vectors = tsne.fit_transform(vectors)

    return vectors, labels

In [23]:
# Graficar los embedddings en 2D
import plotly.graph_objects as go
import plotly.express as px

vecs, labels = reduce_dimensions(w2v_model)

MAX_WORDS=200
fig = px.scatter(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], text=labels[:MAX_WORDS])
fig.show()  # portable: autodetecta Colab / notebook # esto para plotly en colab

In [24]:
# Graficar los embedddings en 3D

vecs, labels = reduce_dimensions(w2v_model,3)

fig = px.scatter_3d(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], z=vecs[:MAX_WORDS,2],text=labels[:MAX_WORDS])
fig.update_traces(marker_size = 2)
fig.show()  # portable: autodetecta Colab / notebook # esto para plotly en colab

In [25]:
# También se pueden guardar los vectores y labels como tsv para graficar en
# http://projector.tensorflow.org/


vectors = np.asarray(w2v_model.wv.vectors)
labels = list(w2v_model.wv.index_to_key)

np.savetxt("vectors.tsv", vectors, delimiter="\t")

with open("labels.tsv", "w") as fp:
    for item in labels:
        fp.write("%s\n" % item)

### Consigna del desafío 2

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado**

Recuerden que su notebook de entrega debe poder correrse de inicio a fin sin la aparición de errores.

- Crear sus propios vectores con Gensim basado en lo visto en clase con un corpus propio (revisar enlaces sugeridos en clase 2 sobre opciones de dataset)
- Elegir términos de interés y buscar términos más similares y menos similares.
- Realizar una reduccion de dimensionalidad a los embeddings, llevándolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de términos (variable MAX_WORDS) de forma tal que la visualización sea adecuada.
- Inspeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)

## Desafío 2: Custom embeddings con Gensim

El objetivo es entrenar embeddings propios con Word2Vec (Gensim) sobre un corpus elegido, estudiar la similaridad entre términos, proyectar los vectores a 2D y analizar los grupos de palabras que se forman.

**Corpus elegido: discursos de la Reserva Federal de EE.UU. (Fed).**

Elijo este corpus por tres razones. Primero, soy economista, así que trabajar sobre política monetaria me permite entender mejor el ejercicio: puedo juzgar si los términos que el modelo asocia tienen sentido económico real, en vez de analizar un dominio que no conozco. Segundo, me sirve para mi tesis de doctorado: aprender a aplicar embeddings sobre discursos de política económica es una técnica que puedo reutilizar en mi investigación. Tercero, es un corpus con vocabulario técnico y temática concentrada (inflación, empleo, tasas, mercados financieros), lo que favorece que los agrupamientos de palabras sean interpretables.

Antes de entrenar, agrego un bloque de análisis exploratorio del corpus. La consigna no lo pide, pero necesito conocer su tamaño y la frecuencia de los términos que voy a estudiar: un término que aparece pocas veces produce un embedding poco confiable, así que conviene verificarlo antes de elegir los términos de interés.

In [26]:
# Descarga directa del CSV con los comunicados y actas del FOMC (2000-2026)
url = "https://raw.githubusercontent.com/vtasca/fed-statement-scraping/master/communications.csv"
df = pd.read_csv(url)

print(f"Documentos cargados: {len(df)}")
print(f"Columnas: {list(df.columns)}\n")

print("Tipos de documento:")
print(df['Type'].value_counts(), "\n")

print(f"Rango de fechas: {df['Date'].min()} a {df['Date'].max()}")
print(f"Documentos sin texto: {df['Text'].isna().sum()}")

df.head(3)

Documentos cargados: 467
Columnas: ['Date', 'Release Date', 'Type', 'Text']

Tipos de documento:
Type
Minute       242
Statement    225
Name: count, dtype: int64 

Rango de fechas: 2000-02-02 a 2026-07-29
Documentos sin texto: 0


,Date,Release Date,Type,Text
0,2026-07-29,2026-07-29,Statement,The Federal Open Market Committee approved the...
1,2026-06-17,2026-06-17,Statement,The Federal Open Market Committee approved the...
2,2026-06-17,2026-07-08,Minute,Minutes of the Federal Open Market Committee\n...


In [27]:
import numpy as np
import re

# Unimos todo el texto y lo pasamos a minúsculas para contar de forma consistente
textos = df['Text'].tolist()
corpus_completo = " ".join(textos).lower()

# Tokenización simple por palabras (solo letras) para el conteo exploratorio
tokens = re.findall(r'[a-z]+', corpus_completo)

print("=== VOLUMEN DEL CORPUS ===")
print(f"Palabras totales:   {len(tokens):,}")
print(f"Palabras únicas:    {len(set(tokens)):,}")

# Palabras por documento, separando por tipo
df['n_palabras'] = df['Text'].apply(lambda t: len(re.findall(r'[a-z]+', t.lower())))
print("\n=== PALABRAS POR DOCUMENTO (según tipo) ===")
print(df.groupby('Type')['n_palabras'].agg(['mean', 'min', 'max', 'sum']).round(0))

# Frecuencia de términos de interés candidatos
print("\n=== FRECUENCIA DE TÉRMINOS DE INTERÉS ===")
from collections import Counter
frec = Counter(tokens)
candidatos = ['inflation', 'unemployment', 'crisis', 'recession',
              'financial', 'argentina', 'emerging', 'banks', 'debt', 
              'credit', 'swap', 'rate', 'interest', 'monetary', 'fiscal',
              'policy','employment', 'growth', 'deflation', 'stimulus', 'quantitative',
              'easing', 'technological', 'information']
for palabra in candidatos:
    print(f"  {palabra:15s} {frec[palabra]:>6,} apariciones")

=== VOLUMEN DEL CORPUS ===
Palabras totales:   1,823,445
Palabras únicas:    9,477

=== PALABRAS POR DOCUMENTO (según tipo) ===
             mean   min    max      sum
Type                                   
Minute     7172.0  3251  14398  1735689
Statement   390.0    78    920    87756

=== FRECUENCIA DE TÉRMINOS DE INTERÉS ===
  inflation       13,577 apariciones
  unemployment     2,449 apariciones
  crisis             249 apariciones
  recession          159 apariciones
  financial        5,958 apariciones
  argentina           27 apariciones
  emerging           487 apariciones
  banks            2,692 apariciones
  debt             1,376 apariciones
  credit           3,242 apariciones
  swap               962 apariciones
  rate             9,732 apariciones
  interest         1,838 apariciones
  monetary         6,115 apariciones
  fiscal             992 apariciones
  policy           8,610 apariciones
  employment       3,092 apariciones
  growth           6,218 apariciones
  d

#### Lo que muestra el output

**Volumen.** El corpus tiene 1.817.891 palabras y un vocabulario de 9.468 términos únicos. Supera el orden de magnitud (~1 millón de palabras) a partir del cual Word2Vec entrena embeddings razonables, así que decido trabajar solo con este corpus.

**Composición.** Las actas (Minutes) aportan 1.730.310 palabras (95% del total) y los comunicados (Statements) solo 87.581 (5%). El corpus es, en la práctica, el texto de las actas: deliberaciones técnicas extensas. Los embeddings van a reflejar ese registro, no el lenguaje de los comunicados.

**Términos de interés.** La frecuencia de los candidatos va de 27 a 13.523 apariciones. Los términos económicos centrales (inflation, monetary, financial, credit, banks, unemployment) aparecen miles de veces y tendrán embeddings confiables. En cambio, `argentina` aparece solo 27 veces, insuficiente para un embedding estable, por lo que lo descarto como término de interés. Para estudiar cómo el corpus trata a las economías en desarrollo uso `emerging` (487 apariciones), que sí tiene volumen. Esta verificación previa evita elegir términos que producirían vecinos sin sentido.

### Análisis de bigramas y trigramas

Antes de preprocesar y entrenar, analizo los bigramas y trigramas más frecuentes del corpus. En economía muchos conceptos son compuestos por dos o tres palabras (por ejemplo "quantitative easing", "labor market", "federal funds rate"), y Word2Vec por defecto trata cada palabra como un token independiente, con lo que perdería estos conceptos como unidad.

El objetivo es doble: entender qué términos compuestos dominan efectivamente este corpus (y no los que yo supongo), y decidir con evidencia si conviene unir esos compuestos en un solo token durante el preprocesamiento. Filtro las stopwords antes de contar, porque de lo contrario los n-gramas más frecuentes serían combinaciones de relleno ("of the", "in the") sin valor interpretativo.

In [28]:
import nltk
from nltk.corpus import stopwords
from nltk import ngrams
from collections import Counter
import re

nltk.download('stopwords', quiet=True)
stop_en = set(stopwords.words('english'))

# Tokenizamos el corpus completo y sacamos stopwords, para que los n-gramas
# muestren conceptos reales y no relleno ("of the", "in the")
tokens_limpios = [w for w in re.findall(r'[a-z]+', corpus_completo) if w not in stop_en]

# Bigramas y trigramas más frecuentes
bigramas = Counter(ngrams(tokens_limpios, 2))
trigramas = Counter(ngrams(tokens_limpios, 3))

print("=== TOP 20 BIGRAMAS ===")
for (w1, w2), n in bigramas.most_common(20):
    print(f"  {w1} {w2:20s} {n:>5,}")

print("\n=== TOP 20 TRIGRAMAS ===")
for (w1, w2, w3), n in trigramas.most_common(20):
    print(f"  {w1} {w2} {w3:25s} {n:>5,}")

=== TOP 20 BIGRAMAS ===
  federal reserve              6,007
  open market               4,180
  board governors            4,009
  federal funds                3,950
  funds rate                 3,787
  monetary policy               3,549
  labor market               3,033
  economic activity             2,776
  intermeeting period               2,721
  reserve bank                 2,657
  inflation expectations         2,142
  target range                1,894
  market committee            1,866
  federal open                 1,858
  monetary affairs              1,841
  longer run                  1,781
  unemployment rate                 1,753
  affairs board                1,752
  division monetary             1,749
  market conditions           1,744

=== TOP 20 TRIGRAMAS ===
  federal funds rate                      3,743
  federal reserve bank                      2,429
  federal open market                    1,858
  open market committee                 1,856
  division monet

#### Lo que muestra el output

Los n-gramas confirman que el corpus está dominado por conceptos económicos compuestos. Entre los bigramas y trigramas aparecen los términos técnicos esperados de política monetaria: "federal funds rate" (3.727), "monetary policy" (3.541), "labor market" (3.022), "economic activity" (2.768), "inflation expectations" (2.134), "unemployment rate" (1.745), "mortgage backed securities" (1.149). Todos son conceptos que pierden significado si se parten en palabras sueltas: "funds rate" o "labor market" son unidades semánticas, no dos palabras independientes.

Aparece también un segundo grupo que no anticipaba: n-gramas institucionales y de formato, propios de las actas. "Board governors", "division monetary affairs", "reserve bank new york", "federal open market committee", "president federal reserve" son fórmulas administrativas que se repiten en cada acta (nombres de áreas, cargos, encabezados). Su alta frecuencia no refleja contenido económico sino la estructura repetitiva de los documentos.

Conclusión para el preprocesamiento: hay suficientes conceptos compuestos con valor económico como para justificar unir bigramas (y algunos trigramas) en tokens únicos mediante detección automática de frases. Al mismo tiempo, la presencia de fórmulas institucionales repetidas sugiere que parte del vocabulario más frecuente es ruido estructural, algo a tener en cuenta al interpretar los embeddings.

### Preprocesamiento

El ejemplo de la cátedra aplica un preprocesamiento mínimo (minúsculas y separación en palabras) y entrena directamente. Sobre esa base agrego dos pasos, justificados en el análisis previo:

1. **Partir las actas en oraciones.** A diferencia de las canciones del ejemplo (que ya vienen partidas por verso), mis documentos son párrafos largos. Los separo en oraciones para que el contexto de cada palabra sean las palabras que la rodean de verdad, y no todo el documento.

2. **Unir bigramas frecuentes en un solo token.** El análisis de n-gramas mostró que el corpus está lleno de conceptos económicos compuestos ("federal funds rate", "labor market", "monetary policy"). Si Word2Vec los tratara como palabras sueltas, perdería esos conceptos como unidad. Uso la detección automática de frases de Gensim para unirlos.

Mantengo el resto del preprocesamiento acotado: minúsculas, quito puntuación, números y los caracteres mal codificados que detecté en la carga. No elimino stopwords de forma manual, porque el propio Word2Vec les baja el peso durante el entrenamiento y la detección de bigramas ya filtra 
las combinaciones de relleno.

In [29]:
import re
import nltk
from nltk.tokenize import sent_tokenize
from gensim.models.phrases import Phrases, ENGLISH_CONNECTOR_WORDS

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def limpiar(texto):
    # Arreglar caracteres mal codificados y normalizar
    texto = texto.encode('latin-1', 'ignore').decode('utf-8', 'ignore')
    texto = texto.lower()
    return texto

# 1) Partir cada documento en oraciones y tokenizar cada oración en palabras
oraciones = []
for doc in df['Text']:
    for sent in sent_tokenize(limpiar(doc)):
        palabras = re.findall(r'[a-z]+', sent)   # solo palabras de letras (sin números ni puntuación)
        if len(palabras) >= 3:                   # descartamos oraciones muy cortas
            oraciones.append(palabras)

print(f"Oraciones totales: {len(oraciones):,}")
print("Ejemplo de oración tokenizada:")
print(oraciones[10])

# 2) Detectar y unir bigramas frecuentes ("federal funds" -> "federal_funds")
bigram = Phrases(oraciones, min_count=20, threshold=10,
                 connector_words=ENGLISH_CONNECTOR_WORDS)
oraciones_bigram = [bigram[o] for o in oraciones]

# Aplicamos una segunda pasada para capturar algunos trigramas
# ("federal_funds" + "rate" -> "federal_funds_rate")
trigram = Phrases(oraciones_bigram, min_count=20, threshold=10,
                  connector_words=ENGLISH_CONNECTOR_WORDS)
oraciones_final = [trigram[o] for o in oraciones_bigram]

# Verificamos que se hayan formado los compuestos
ejemplo = [tok for o in oraciones_final for tok in o if '_' in tok]
print(f"\nTokens compuestos formados (ejemplos): {list(dict.fromkeys(ejemplo))[:15]}")

Oraciones totales: 58,410
Ejemplo de oración tokenizada:
['the', 'committee', 'reaffirmed', 'its', 'policy', 'of', 'maintaining', 'ample', 'reserves', 'in', 'the', 'banking', 'system']

Tokens compuestos formados (ejemplos): ['federal_open_market', 'committee_decided_to_maintain_the_target', 'range_for_the_federal_funds', 'federal_reserve', 's_dual_mandate', 'ample_reserves', 'banking_system', 'economic_activity', 'expanding_at_a_solid_pace', 'conflict_in_the_middle_east', 'productivity_growth', 'job_gains', 'unemployment_rate_has', 'changed_little', 'remains_elevated']


#### Lo que muestra el output y decisión

Se generaron 58.233 oraciones y la tokenización es correcta. La detección de compuestos capturó bien varios conceptos económicos reales: `federal_reserve`, `federal_open_market`, `economic_activity`, `unemployment_rate`, `productivity_growth`, `middle_east`. Pero también se formaron tokens defectuosos: `committee_decided_to_maintain_the_target`, `range_for_the_federal_funds`, `expanding_at_a_solid_pace`. Estos no son conceptos, son frases enteras pegadas. El problema tiene dos causas: el umbral (`threshold`) quedó demasiado permisivo, y al aplicar la detección dos veces (bigramas y luego trigramas) la segunda pasada encadenó secuencias ya formadas. Las actas del FOMC repiten frases casi textuales en 
cada comunicado, así que esas secuencias largas superan el umbral y se pegan. Un token de este tipo aparece muy pocas veces y produce un embedding inútil, lo contrario de lo buscado. Además, `s_dual_mandate` muestra que el apóstrofe de las contracciones ("Fed's") dejó una "s" suelta que 
se pegó a otro token.

Por esto decido quedarme solo con bigramas y subir los umbrales de detección. Con una sola pasada y umbrales más exigentes, se capturan los compuestos con asociación fuerte y frecuente (`federal_funds`, `labor_market`, `monetary_policy`) y se evita el encadenamiento de frases. La pérdida de trigramas como `federal_funds_rate` es menor: para los embeddings, la diferencia entre ese trigrama y `federal_funds` + `rate` por separado es despreciable. También agrego un filtro que descarta los tokens de una sola letra, para eliminar los residuos de las contracciones.

In [30]:

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def limpiar(texto):
    texto = texto.encode('latin-1', 'ignore').decode('utf-8', 'ignore')
    return texto.lower()

# 1) Partir en oraciones y tokenizar (solo palabras de 2+ letras, sin números)
oraciones = []
for doc in df['Text']:
    for sent in sent_tokenize(limpiar(doc)):
        palabras = [w for w in re.findall(r'[a-z]+', sent) if len(w) > 1]
        if len(palabras) >= 3:
            oraciones.append(palabras)

print(f"Oraciones totales: {len(oraciones):,}")

# 2) Detectar y unir SOLO bigramas frecuentes, con umbrales exigentes
bigram = Phrases(oraciones, min_count=30, threshold=50,
                 connector_words=ENGLISH_CONNECTOR_WORDS)
oraciones_final = [bigram[o] for o in oraciones]

# Verificamos qué compuestos se formaron y con qué frecuencia
from collections import Counter
compuestos = Counter(tok for o in oraciones_final for tok in o if '_' in tok)
print(f"\nCompuestos distintos formados: {len(compuestos)}")
print("Top 20 compuestos por frecuencia:")
for tok, n in compuestos.most_common(20):
    print(f"  {tok:30s} {n:>5,}")

Oraciones totales: 58,410

Compuestos distintos formados: 668
Top 20 compuestos por frecuencia:
  target_range                   1,894
  new_york                       1,607
  mortgage_backed                1,157
  maximum_employment             1,111
  little_changed                 1,087
  real_gdp                       1,023
  research_and_statistics          934
  balance_sheet                    863
  director_division                802
  associate_director               769
  united_states                    700
  return_to_text                   604
  unanimous_vote                   547
  vice_president                   538
  basis_points                     514
  fixed_investment                 457
  motor_vehicles                   457
  picked_up                        446
  billion_per                      429
  special_adviser                  425


#### Lo que muestra el output

Con los umbrales corregidos se formaron 666 compuestos, todos bigramas legítimos de dos palabras, sin frases pegadas. Se preservaron los conceptos económicos que buscaba: `target_range`, `mortgage_backed`, `maximum_employment`, `real_gdp`, `balance_sheet`, `basis_points`, `fixed_investment`.

Reaparece el ruido institucional ya identificado en el análisis de n-gramas: `research_and_statistics`, `associate_director`, `vice_president`, `special_adviser`, `return_to_text`. Son fórmulas administrativas de las actas (cargos, áreas, notas al pie), no vocabulario económico. Entreno un 
único modelo sobre el corpus completo, sin filtrar estos términos, porque no forman parte de mis términos de interés y no afectan el análisis de similaridad. El impacto de este ruido se ve en la visualización 2D, donde estos términos supongo que tenderan a agruparse: por eso, en esa instancia voy a comparar el gráfico con y sin los términos institucionales, para mostrar cómo su presencia afecta la lectura de los grupos económicos. El filtrado se hará solo sobre la visualización, digo esto porque en un primer momento pensé hacerlo sobre el entrenamiento.

### Entrenamiento de Word2Vec

Uso como base los hiperparámetros del ejemplo de la cátedra, porque son un punto de partida ya validado y me permiten mantenerme dentro de lo visto en clase:

- **Skip-gram (`sg=1`):** predice el contexto a partir de cada palabra. Funciona bien en corpus medianos y representa mejor las palabras poco frecuentes, que es mi caso.
- **`vector_size=300`:** dimensión de los embeddings. Es el valor estándar y el que usa el ejemplo.
- **`min_count=5`:** descarto las palabras que aparecen menos de 5 veces, porque con tan pocas apariciones el embedding es poco confiable (mismo criterio que apliqué al descartar términos de baja frecuencia en el análisis exploratorio).
- **`negative=20` y `epochs=20`:** parámetros de optimización y cantidad de pasadas, los mantengo como en el ejemplo.

El único parámetro que modifico es la **ventana de contexto**, que subo de 2 a 5. La cátedra usa `window=2` porque su corpus son canciones, donde cada línea es corta. Mi corpus son actas del FOMC, con oraciones más largas y sintaxis compleja, donde las palabras relacionadas suelen estar más 
separadas dentro de la oración. Una ventana más amplia captura mejor esas relaciones. Es el único cambio, y está fundamentado en la diferencia entre ambos corpus.

In [31]:

# Callback para ver el loss en cada época (igual que el ejemplo de la cátedra)
class LossCallback(CallbackAny2Vec):
    def __init__(self):
        self.epoch = 0
        self.loss_previo = 0
    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        delta = loss if self.epoch == 0 else loss - self.loss_previo
        print(f"Época {self.epoch:2d} - loss: {delta:.0f}")
        self.epoch += 1
        self.loss_previo = loss

# Instanciamos el modelo. Único cambio respecto del ejemplo: window=5
w2v = Word2Vec(
    min_count=5,       # palabra debe aparecer al menos 5 veces
    window=5,          # 5 palabras de contexto a cada lado (subido de 2: oraciones más largas)
    vector_size=300,   # dimensión de los embeddings
    negative=20,       # negative sampling
    workers=4,         # núcleos para entrenar (tenés varios)
    sg=1               # Skip-gram
)

w2v.build_vocab(oraciones_final)

print("Documentos (oraciones) en el corpus:", w2v.corpus_count)
print("Palabras distintas en el vocabulario:", len(w2v.wv.index_to_key))

Documentos (oraciones) en el corpus: 58410
Palabras distintas en el vocabulario: 6064


#### Lo que muestra el output

El vocabulario quedó en 6.055 palabras, frente a las 9.468 únicas detectadas en la exploración. El `min_count=5` descartó unas 3.400 palabras de frecuencia menor a 5 (typos, términos aislados, nombres sueltos), dejando solo las que tienen evidencia suficiente para un embedding confiable. 
Es el mismo criterio de frecuencia mínima aplicado en el análisis exploratorio.

In [32]:
w2v.train(
    oraciones_final,
    total_examples=w2v.corpus_count,
    epochs=20,
    compute_loss=True,
    callbacks=[LossCallback()]
)

Época  0 - loss: 4542038
Época  1 - loss: 3866470
Época  2 - loss: 3522438
Época  3 - loss: 3504316
Época  4 - loss: 3318946
Época  5 - loss: 3216098
Época  6 - loss: 3197510
Época  7 - loss: 3249580
Época  8 - loss: 3173496
Época  9 - loss: 3097136
Época 10 - loss: 3026176
Época 11 - loss: 2990960
Época 12 - loss: 3049080
Época 13 - loss: 3022132
Época 14 - loss: 3090776
Época 15 - loss: 3053016
Época 16 - loss: 3091504
Época 17 - loss: 3106316
Época 18 - loss: 3221268
Época 19 - loss: 3308048


(24895222, 33617000)

#### Lo que muestra el output

El loss desciende de 4.66M en la primera época a 2.82M en la última, con una caída pronunciada al inicio (el modelo aprende primero las relaciones más marcadas) y un descenso más gradual después. En las últimas épocas el loss oscila levemente en lugar de bajar de forma monótona, lo cual es esperable en Word2Vec: el entrenamiento usa muestreo aleatorio (negative sampling), por lo que el loss por época tiene ruido. Lo relevante es la tendencia general descendente, que confirma que el modelo entrenó correctamente. Las 20 épocas son suficientes; el entrenamiento convergió.

### Términos similares y opuestos

Elijo cinco términos de interés que cubren distintas dimensiones del corpus:

- **`inflation`** (13.523 apariciones): el término macroeconómico central del corpus y objetivo principal de la Fed. Funciona como caso de control: si el modelo entrenó bien, sus vecinos deberían ser económicamente coherentes.
- **`crisis`** (249): término de menor frecuencia pero ligado a mi interés por los episodios de crisis financiera. Permite ver cómo el corpus los enmarca léxicamente.
- **`easing`** (761): parte de "quantitative easing", instrumento central de la política monetaria no convencional posterior a 2008.
- **`unemployment`** (2.440): la variable canónica del mercado de trabajo. Es el fenómeno económico que me interesa —la dinámica del empleo en la economía— y no el objetivo estatutario de la Fed.
- **`emerging`** (487): lo uso en reemplazo de "argentina" (descartado por baja frecuencia) para estudiar cómo el corpus trata a las economías en desarrollo.

Para cada término busco los más similares (positive) y los menos similares (negative), siguiendo el método del ejemplo de la cátedra.

#### Verificación previa del vocabulario

Antes de consultar los términos verifico cuáles quedaron efectivamente en el vocabulario del modelo. Este paso es necesario porque el preprocesamiento con `Phrases` no garantiza que todos los bigramas del texto crudo sobrevivan como tokens.

In [33]:
# Bigramas (tokens con "_") que sí quedaron en el vocabulario del modelo
bigramas_vocab = [(w, w2v.wv.get_vecattr(w, 'count'))
                  for w in w2v.wv.index_to_key if '_' in w]
bigramas_vocab.sort(key=lambda x: x[1], reverse=True)

print(f"Bigramas en el vocabulario: {len(bigramas_vocab)}\n")
print("Top 25 por frecuencia:")
for w, n in bigramas_vocab[:25]:
    print(f"  {w:28s} {n:>5,}")

Bigramas en el vocabulario: 649

Top 25 por frecuencia:
  target_range                 1,894
  new_york                     1,607
  mortgage_backed              1,157
  maximum_employment           1,111
  little_changed               1,087
  real_gdp                     1,023
  research_and_statistics        934
  balance_sheet                  863
  director_division              802
  associate_director             769
  united_states                  700
  return_to_text                 604
  unanimous_vote                 547
  vice_president                 538
  basis_points                   514
  motor_vehicles                 457
  fixed_investment               457
  picked_up                      446
  billion_per                    429
  special_adviser                425
  notation_vote                  409
  job_gains                      408
  did_not                        395
  san_francisco                  388
  resource_utilization           382


In [34]:
# Verificamos qué términos de interés están en el vocabulario antes de consultarlos
candidatos = ["inflation", "crisis", "easing", "labor_market",
              "maximum_employment", "unemployment", "emerging"]

print(f"{'término':22s} {'¿en vocab?':12s} {'frecuencia':>10s}")
print("-" * 46)
for t in candidatos:
    if t in w2v.wv:
        print(f"{t:22s} {'sí':12s} {w2v.wv.get_vecattr(t, 'count'):>10,}")
    else:
        print(f"{t:22s} {'NO':12s} {'-':>10s}")

término                ¿en vocab?   frecuencia
----------------------------------------------
inflation              sí               13,576
crisis                 sí                  249
easing                 sí                  764
labor_market           NO                    -
maximum_employment     sí                1,111
unemployment           sí                2,265
emerging               sí                  429


#### Problema detectado: término inexistente en el vocabulario

Mi selección inicial incluía `labor_market`, pero ese token no está en el vocabulario del modelo, pese a que "labor market" es uno de los bigramas más frecuentes del corpus en texto crudo (3.022 apariciones).

La causa está en cómo funciona la detección de bigramas: `Phrases` no une dos palabras por su frecuencia absoluta, sino por su fuerza de asociación, es decir, cuánto aparecen juntas respecto de cuánto aparecen por separado. "Market" se combina con muchas otras palabras en este corpus ("open market", "market conditions", "market committee", "money market"), por lo que la asociación específica "labor"+"market" no es lo bastante exclusiva y no superó el umbral que fijé (`threshold=50`). Es un efecto colateral asumido de esa decisión: el umbral alto me protegió de las frases pegadas, a costa de perder algunos bigramas frecuentes pero poco exclusivos.

**Decisión.** Reemplazo `labor_market` por `unemployment` (2.440 apariciones). Descarto `maximum_employment`, que sí está en el vocabulario, porque es un término jurídico —el objetivo estatutario de la Fed— y no una variable del mercado de trabajo. Mi interés es la dinámica del empleo como fenómeno económico, no la definición legal del mandato.

In [35]:
terminos = ["inflation", "crisis", "easing", "unemployment", "emerging"]

for t in terminos:
    print("=" * 70)
    if t not in w2v.wv:
        print(f"'{t}' no está en el vocabulario.")
        continue
    print(f"TÉRMINO: {t}   (frecuencia: {w2v.wv.get_vecattr(t, 'count'):,})")
    print("-" * 70)
    print("  MÁS similares:")
    for palabra, sim in w2v.wv.most_similar(positive=[t], topn=8):
        print(f"    {palabra:22s} {sim:.3f}")
    print("  MENOS similares (opuestos):")
    for palabra, sim in w2v.wv.most_similar(negative=[t], topn=5):
        print(f"    {palabra:22s} {sim:.3f}")
    print()

TÉRMINO: inflation   (frecuencia: 13,576)
----------------------------------------------------------------------
  MÁS similares:
    expectations           0.456
    headline               0.453
    disinflationary        0.450
    core                   0.449
    unmooring              0.435
    objective              0.434
    crept                  0.423
    quiescent              0.421
  MENOS similares (opuestos):
    scheme                 0.063
    constraints            0.051
    register               0.024
    exercises              0.022
    extended               0.021

TÉRMINO: crisis   (frecuencia: 249)
----------------------------------------------------------------------
  MÁS similares:
    gfc                    0.475
    percentile             0.372
    interaction            0.359
    th                     0.351
    norms                  0.351
    pre                    0.350
    valley                 0.349
    silicon                0.343
  MENOS similares (opu

#### Análisis de términos similares y opuestos

**inflation.** Los vecinos son vocabulario técnico de política monetaria: `headline` y `core` son los dos tipos de inflación que monitorea la Fed (total y subyacente), `expectations` remite a las expectativas de inflación, y aparecen también `disinflationary`, `objective` (la meta del 2%) y `unmooring` (el desanclaje de expectativas). Como caso de control confirma que el modelo entrenó bien: un término frecuente y central produce vecinos económicamente correctos.

**crisis.** El vecino más cercano es `gfc` (Global Financial Crisis), la sigla de la crisis de 2008. 
Aparecen también `silicon` y `valley`, que remiten a Silicon Valley Bank (2023), y `pandemic` (2020). 
El embedding agrupa entonces tres episodios distintos separados en el tiempo, porque el corpus los 
describe con vocabulario similar. También aparece `onset`, el inicio de un episodio. El modelo 
capturó la noción de crisis de forma transversal al período completo del corpus.

**easing.** El vecino más similar de `easing` (relajar la política) es `tightening` (endurecerla), su antónimo. Esto ocurre porque Word2Vec mide contexto, no significado: ambos términos aparecen en las mismas construcciones sintácticas. El modelo aprende que son palabras del mismo tipo —decisiones de política monetaria— no que signifiquen lo mismo. Aparece también `accommodative`, el término técnico para una política expansiva.

**unemployment.** Los vecinos capturan tres dimensiones del mercado de trabajo. Primero, sinónimos 
directos: `jobless` y `joblessness`. Segundo, los indicadores técnicos con que se mide el mercado 
laboral: `lfpr` (labor force participation rate), `epop` (employment-to-population ratio) y 
`force_participation`. Tercero —y es lo más relevante— aparecen `african_americans` e `hispanics`, 
lo que muestra que el corpus no trata el desempleo solo como un agregado macroeconómico, sino que 
discute su desagregación por grupo demográfico. Esto refleja el giro de la Fed hacia una lectura 
distributiva del empleo, más explícita a partir de su marco de 2020.

**emerging.** Los vecinos son `emes`/`eme` (siglas de Emerging Market Economies), `economies`, y regiones como `asian`, `asia` y `latin`. El modelo ubica a las economías emergentes como categoría y las asocia a las regiones donde se concentran, cubriendo el interés original por cómo el corpus trata a los países en desarrollo.

**Sobre los términos opuestos.** En todos los casos los "menos similares" son palabras de contexto lejano y poco interpretables, no antónimos. Es esperable en Word2Vec: la dirección opuesta de un vector no corresponde a un significado contrario sino a contextos sin relación. El análisis útil está en los términos similares.

### Reducción de dimensionalidad y visualización

Los embeddings tienen 300 dimensiones, que no se pueden visualizar directamente. Para poder 
inspeccionarlos hay que proyectarlos a 2D. Las decisiones:

- **Técnica: t-SNE.** Descarto PCA porque, al ser una proyección lineal que maximiza varianza 
  global, tiende a concentrar los puntos en el centro y los grupos no se distinguen. t-SNE preserva 
  la estructura local: busca que las palabras cercanas en el espacio original queden cercanas en la 
  proyección. Como la consigna pide identificar grupos de palabras, necesito una técnica que los 
  haga visibles, y t-SNE es la adecuada. Es también la que usa el ejemplo de la cátedra.

- **`MAX_WORDS = 200`.** Es un trade-off entre cobertura y legibilidad: con pocas palabras se ven 
  pocos grupos, y con muchas las etiquetas se superponen y el gráfico deja de ser interpretable. 
  Empiezo con 200 (el valor del ejemplo de la cátedra) y ajusto si la visualización no resulta 
  clara. Las palabras se toman ordenadas por frecuencia, de modo que se grafican los términos más 
  representativos del corpus.

- **Dos visualizaciones: con y sin términos institucionales.** En el análisis de n-gramas detecté 
  que buena parte del vocabulario más frecuente no es económico sino administrativo (cargos, áreas, 
  notas al pie). Como esos términos son frecuentes, van a entrar en el gráfico y pueden distorsionar 
  la lectura de los grupos. Grafico primero sin filtrar, para mostrar el problema, y luego filtrando 
  esos términos mediante una lista manual construida a partir de lo que ya identifiqué. Uso lista 
  manual porque es transparente y verificable, y porque el objetivo acá es la interpretación, no la 
  sofisticación del criterio de filtrado. El filtrado es solo sobre la visualización: el modelo 
  entrenado es el mismo en ambos casos.

Gráfico A (sin filtrar)

In [36]:
from sklearn.manifold import TSNE
import numpy as np
import plotly.express as px

MAX_WORDS = 200

# Tomamos las MAX_WORDS más frecuentes del vocabulario
palabras = w2v.wv.index_to_key[:MAX_WORDS]
vectores = np.array([w2v.wv[w] for w in palabras])

print(f"Palabras a graficar: {len(palabras)}")
print(f"Dimensión original: {vectores.shape}")

# Proyección a 2D con t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=20, init='pca')
coords = tsne.fit_transform(vectores)

print(f"Dimensión proyectada: {coords.shape}")

# Gráfico interactivo
fig = px.scatter(
    x=coords[:, 0], y=coords[:, 1], text=palabras,
    title=f"Embeddings FOMC proyectados a 2D — {MAX_WORDS} términos más frecuentes (sin filtrar)"
)
fig.update_traces(textposition='top center', marker=dict(size=6))
fig.update_layout(height=800, width=1100)
fig.show()

Palabras a graficar: 200
Dimensión original: (200, 300)
Dimensión proyectada: (200, 2)


#### Lo que muestra el gráfico sin filtrar

La proyección permite identificar algunos grupos con sentido económico:

- **Política monetaria** (izquierda-centro): `inflation`, `expectations`, `objective`, `stability`, 
  `price`, `percent`, `rates`, `interest`, `funds`, `target_range`, `unemployment`.
- **Actividad económica** (derecha): `investment`, `housing`, `sales`, `consumer`, `demand`, `labor`, 
  `employment`, `growth`, `economy`, `activity`, `outlook`, `sector`, `business`.
- **Instrumentos financieros** (extremo izquierdo): `securities`, `treasury`, `agencies`, `yields`.
- **Institucional/operativo** (zona inferior): `federal`, `reserve`, `new_york`, `system`, `account`, 
  `committee`, `meeting`, `manager`, y más abajo `secretary`, `governors`, `division`, `senior`, `mr`.

Pero el gráfico presenta dos problemas de legibilidad, y ambos se ven directamente en la imagen.

**Primero, el ruido institucional que ya había detectado en el análisis de n-gramas.** Los términos 
administrativos (`governors`, `division`, `secretary`, `senior`, `associate`, `mr`) ocupan un sector 
propio del gráfico. No aportan información económica: son fórmulas de las actas que se repiten en 
cada documento.

**Segundo, un problema que no había anticipado: las palabras funcionales dominan el centro.** Todo el 
sector medio del gráfico está ocupado por stopwords (`will`, `might`, `could`, `would`, `likely`, 
`have`, `were`, `in`, `on`, `at`, `with`, `and`, `also`, `these`, `some`, `many`, `not`, `they`). Como 
son las palabras más frecuentes del corpus, ocupan la mayoría de los 200 lugares disponibles y empujan 
los términos con contenido hacia los bordes.

Esta es la consecuencia visible de haber decidido no eliminar stopwords en el preprocesamiento. Esa 
decisión fue correcta para el entrenamiento —Word2Vec les baja el peso por sí mismo y en el análisis 
de términos similares no interfirieron— pero no lo es para la visualización, donde la selección se 
hace por frecuencia y el espacio es limitado.

**Decisión.** Filtro ambos tipos de términos para la segunda visualización: las stopwords mediante la 
lista estándar de NLTK, y los términos institucionales mediante una lista manual construida a partir 
de lo identificado en el análisis de n-gramas. El filtrado se aplica solo a la selección de palabras a 
graficar; el modelo entrenado es el mismo.

![Gráfico sin filtrar](img/foms_2D.png)

Gráfico B (filtrado)

![Gráfico filtrado](img/filttrado.png)

In [37]:
from nltk.corpus import stopwords

stop_en = set(stopwords.words('english'))

# Términos institucionales identificados en el análisis de n-gramas y en el gráfico anterior
institucionales = {
    'mr', 'secretary', 'governors', 'division', 'senior', 'associate', 'manager',
    'board_governors', 'associate_director', 'director_division', 'vice_president',
    'special_adviser', 'division_monetary', 'monetary_affairs', 'research_and_statistics',
    'return_to_text', 'notation_vote', 'unanimous_vote', 'deputy', 'assistant',
    'staff', 'committee', 'meeting', 'chairman', 'president', 'system_open',
    'open_market_account', 'account', 'minutes', 'attended', 'members'
}

def es_valida(w):
    return (w not in stop_en) and (w not in institucionales) and (len(w) > 2)

# Recorremos el vocabulario por frecuencia y tomamos las primeras MAX_WORDS válidas
palabras_f = [w for w in w2v.wv.index_to_key if es_valida(w)][:MAX_WORDS]
vectores_f = np.array([w2v.wv[w] for w in palabras_f])

print(f"Palabras a graficar (filtradas): {len(palabras_f)}")
print(f"Primeras 20: {palabras_f[:20]}")

tsne_f = TSNE(n_components=2, random_state=42, perplexity=20, init='pca')
coords_f = tsne_f.fit_transform(vectores_f)

fig = px.scatter(
    x=coords_f[:, 0], y=coords_f[:, 1], text=palabras_f,
    title=f"Embeddings FOMC en 2D — {MAX_WORDS} términos más frecuentes (filtrado)"
)
fig.update_traces(textposition='top center', marker=dict(size=6))
fig.update_layout(height=800, width=1100)
fig.show()

Palabras a graficar (filtradas): 200
Primeras 20: ['inflation', 'federal', 'market', 'economic', 'participants', 'rate', 'policy', 'board', 'reserve', 'would', 'growth', 'monetary', 'financial', 'conditions', 'prices', 'remained', 'continued', 'percent', 'term', 'period']


### Identificación e interpretación de grupos

El gráfico filtrado dejó ver grupos con sentido económico, pero sigue teniendo dos problemas: un 
residuo de vocabulario procedimental que las stopwords de NLTK no cubren (verbos deliberativos como 
`indicated`, `suggested`, `judged`; adverbios de matiz como `generally`, `somewhat`; y un clúster 
completo de meses del año), y demasiadas etiquetas superpuestas.

Aplico entonces dos ajustes: amplío el filtro para cubrir esa tercera capa de ruido, y bajo 
`MAX_WORDS` de 200 a 150 para mejorar la legibilidad.

Para destacar los grupos uso dos enfoques complementarios:

1. **Categorías temáticas propias.** Clasifico los términos según categorías económicas que defino a 
   partir de mi conocimiento del dominio: política monetaria, mercado laboral, actividad económica, 
   precios e inflación, e instrumentos y mercados financieros. Esto hace explícita mi lectura del 
   corpus.

2. **Clustering automático (K-Means).** Agrupo los puntos sin intervención, según su proximidad en el 
   espacio proyectado. Sirve como contraste: si los grupos que forma el algoritmo coinciden con mis 
   categorías temáticas, es evidencia de que los embeddings capturaron efectivamente la estructura 
   económica del corpus y no una organización arbitraria.

In [38]:
from nltk.corpus import stopwords
import numpy as np
import plotly.express as px
from sklearn.manifold import TSNE

MAX_WORDS = 150
stop_en = set(stopwords.words('english'))

# --- Capa 1: ruido institucional (identificado en el análisis de n-gramas) ---
institucionales = {
    'mr','secretary','governors','division','senior','associate','manager','deputy','assistant',
    'staff','committee','meeting','chairman','president','minutes','attended','members',
    'board_governors','associate_director','director_division','vice_president','special_adviser',
    'division_monetary','monetary_affairs','research_and_statistics','return_to_text',
    'notation_vote','unanimous_vote','system_open','open_market_account','account','affairs'
}

# --- Capa 2: vocabulario procedimental de las actas (visible en el gráfico anterior) ---
procedimental = {
    'indicated','suggested','judged','reported','appeared','noted','continued','remained','moved',
    'rose','declined','increased','decreased','maintain','support','changes','agreed','discussion',
    'participants','pointed','based','including','likely','anticipated','expected','generally',
    'somewhat','moderately','overall','following','recent','additional','current','future',
    'consistent','appropriate','available','number','range','levels','level','path','part','end',
    'new','next','earlier','since','time','period','month','months','year','years','quarter',
    'first','second','third','fourth','little_changed','well','still','also','although','however',
    'near','low','higher','survey','measures','effects','potential','uncertainty','information',
    'data','indicators','strong','moderate'
}

meses = {'january','february','march','april','may','june','july',
         'august','september','october','november','december'}

excluir = institucionales | procedimental | meses

def es_valida(w):
    return (w not in stop_en) and (w not in excluir) and (len(w) > 2)

palabras_f = [w for w in w2v.wv.index_to_key if es_valida(w)][:MAX_WORDS]
vectores_f = np.array([w2v.wv[w] for w in palabras_f])
print(f"Palabras graficadas: {len(palabras_f)}")

# --- Categorías temáticas ---
categorias = {
    'Política monetaria': {'monetary_policy','policy','fomc','stance','accommodative','target_range',
        'funds','federal_funds','rate','rates','interest','tightening','easing','action','decision',
        'basis_points','longer_run','objective','stability','purchases','operations','balance_sheet'},
    'Mercado laboral': {'unemployment','employment','labor','jobs','wages','payrolls','hiring',
        'maximum_employment','compensation','job_gains','workers','participation'},
    'Actividad económica': {'growth','economic_activity','expansion','output','real_gdp','investment',
        'spending','consumer','household','housing','sector','business','exports','demand','sales',
        'production','firms','economy','economic','pace','activity','outlook'},
    'Precios e inflación': {'inflation','prices','price','core','pce','energy','headline',
        'expectations','disinflationary','deflation','nominal','dollar','currency'},
    'Mercados financieros': {'securities','treasury','mortgage_backed','holdings','yields','spreads',
        'debt','credit','loans','banks','markets','market','financial','equity','transactions',
        'federal_reserve','new_york','system','foreign','domestic','risk','risks'},
}

def categorizar(w):
    for cat, terms in categorias.items():
        if w in terms:
            return cat
    return 'Otros'

etiquetas = [categorizar(w) for w in palabras_f]

# --- Proyección y gráfico ---
tsne = TSNE(n_components=2, random_state=42, perplexity=15, init='pca')
coords = tsne.fit_transform(vectores_f)

fig = px.scatter(
    x=coords[:,0], y=coords[:,1], text=palabras_f, color=etiquetas,
    title=f"Embeddings FOMC en 2D — {MAX_WORDS} términos, coloreados por categoría temática"
)
fig.update_traces(textposition='top center', marker=dict(size=8))
fig.update_layout(height=850, width=1200)
fig.show()

# Cuántos términos quedaron en cada categoría
from collections import Counter
print("\nTérminos por categoría:")
for cat, n in Counter(etiquetas).most_common():
    print(f"  {cat:24s} {n}")

Palabras graficadas: 150



Términos por categoría:
  Otros                    78
  Mercados financieros     21
  Actividad económica      21
  Política monetaria       15
  Precios e inflación      10
  Mercado laboral          5


#### Ajuste de las categorías

El primer gráfico por categorías dejó 78 de 150 términos en "Otros", más de la mitad, lo que 
debilita la lectura. Al revisarlos encuentro dos situaciones distintas:

- **Ruido no cubierto por el filtro anterior:** una capa adicional de vocabulario procedimental que 
  se me había escapado (`messrs`, `office`, `statement`, `respectively`, `readings`, `incoming`) y, 
  sobre todo, verbos y sustantivos de variación (`rise`, `fell`, `decline`, `increase`, `gains`, 
  `elevated`) que las actas usan constantemente para describir movimientos sin aportar contenido 
  temático.

- **Términos económicos que mis categorías no contemplaban:** `government`, `international`, `global`, 
  `liquidity`, `manufacturing`, `production`, `goods`, `recovery`, `mortgage`, `pressures`. Son 
  económicos pero mis listas iniciales quedaron cortas.

Corrijo ambas cosas: amplío el filtro para la capa de ruido restante y amplío las listas de 
categorías para absorber los términos económicos sueltos. Mantengo una categoría "Otros" —no fuerzo 
que todo entre en mis categorías— porque un residuo pequeño es esperable y más honesto que forzar 
clasificaciones.

#### Gráfico enfocado en el mercado de trabajo

La categoría "Mercado laboral" quedó con pocos términos, porque el vocabulario laboral específico no 
está entre los más frecuentes del corpus (dominado por política monetaria e inflación). Como es el 
área que me interesa, agrego una visualización enfocada: en lugar de graficar los términos más 
frecuentes, grafico el vecindario de `unemployment` y términos laborales relacionados. Esto responde 
a la consigna de "buscar pequeños grupos de palabras" e interpretarlos, aplicado a un grupo temático 
específico.

Categorías ampliadas

![Categorías temáticas ampliadas](img/list_ampl.png)

In [39]:
# --- Capa 3 de ruido: verbos y sustantivos de variación, y residuos de formato ---
ruido_extra = {
    'messrs','office','statement','respectively','economist','readings','incoming','intermeeting',
    'remain','continue','toward','regarding','associated','related','relatively','previous','early',
    'seen','less','light','held','addition','net','average','large','solid','substantial','slightly',
    'little','several','factors','might','would','reflecting','developments','conditions',
    'rise','rising','rose','fell','decline','declines','declined','increase','increases','increased',
    'gains','elevated','lower','total','run','long','term','board','federal'
}
excluir = institucionales | procedimental | meses | ruido_extra

def es_valida(w):
    return (w not in stop_en) and (w not in excluir) and (len(w) > 2)

palabras_f = [w for w in w2v.wv.index_to_key if es_valida(w)][:MAX_WORDS]
vectores_f = np.array([w2v.wv[w] for w in palabras_f])

# --- Categorías ampliadas ---
categorias = {
    'Política monetaria': {'monetary_policy','policy','fomc','stance','accommodative','target_range',
        'funds','federal_funds','rate','rates','interest','tightening','easing','action','purchases',
        'basis_points','objective','stability','operations','balance_sheet','open','system',
        'federal_reserve','accommodation','liquidity'},
    'Mercado laboral': {'unemployment','employment','labor','jobs','wages','payrolls','hiring',
        'maximum_employment','compensation','job_gains','workers','participation','jobless',
        'force_participation','lfpr','epop','joblessness'},
    'Actividad económica': {'growth','economic_activity','expansion','output','real_gdp','investment',
        'spending','consumer','household','housing','sector','business','exports','demand','sales',
        'production','firms','economy','economic','pace','activity','outlook','manufacturing','goods',
        'recovery','international','global','economies','domestic','foreign'},
    'Precios e inflación': {'inflation','prices','price','core','pce','energy','headline','expectations',
        'disinflationary','deflation','nominal','real','dollar','currency','currencies','pressures'},
    'Mercados financieros': {'securities','treasury','mortgage_backed','mortgage','holdings','yields',
        'spreads','debt','credit','loans','banks','markets','market','financial','equity',
        'transactions','new_york','risk','risks','downside','government','bank'},
}

etiquetas = [categorizar(w) for w in palabras_f]

tsne = TSNE(n_components=2, random_state=42, perplexity=15, init='pca')
coords = tsne.fit_transform(vectores_f)

fig = px.scatter(x=coords[:,0], y=coords[:,1], text=palabras_f, color=etiquetas,
    title=f"Embeddings FOMC en 2D — {MAX_WORDS} términos por categoría temática (listas ampliadas)")
fig.update_traces(textposition='top center', marker=dict(size=8))
fig.update_layout(height=850, width=1200)
fig.show()

from collections import Counter
print("\nTérminos por categoría:")
for cat, n in Counter(etiquetas).most_common():
    print(f"  {cat:24s} {n}")


Términos por categoría:
  Otros                    62
  Actividad económica      29
  Mercados financieros     22
  Política monetaria       19
  Precios e inflación      13
  Mercado laboral          5


Gráfico enfocado en mercado de trabajo

![Vecindario del mercado de trabajo](img/vec_sem_MT.png)

In [40]:
# Vecindario de unemployment: sus términos más similares, más términos laborales del vocabulario
semilla = ['unemployment', 'employment', 'labor', 'compensation', 'maximum_employment']

vecinos = set()
for s in semilla:
    if s in w2v.wv:
        vecinos.add(s)
        for w, _ in w2v.wv.most_similar(s, topn=12):
            vecinos.add(w)

vecinos = [w for w in vecinos if w in w2v.wv]
vec_lab = np.array([w2v.wv[w] for w in vecinos])
print(f"Términos en el vecindario laboral: {len(vecinos)}")

tsne_lab = TSNE(n_components=2, random_state=42,
                perplexity=min(15, len(vecinos)-1), init='pca')
coords_lab = tsne_lab.fit_transform(vec_lab)

fig = px.scatter(x=coords_lab[:,0], y=coords_lab[:,1], text=vecinos,
    title="Vecindario semántico del mercado de trabajo en el corpus FOMC")
fig.update_traces(textposition='top center', marker=dict(size=9, color='steelblue'))
fig.update_layout(height=750, width=1000)
fig.show()

print("\nTérminos incluidos:")
print(", ".join(sorted(vecinos)))

Términos en el vecindario laboral: 56



Términos incluidos:
achieve_maximum, african_americans, age, average_hourly, compensation, cost_index, crept, disappointingly, dual, dual_mandate, dynamism, eci, employment, epop, force_participation, goals, hispanics, hourly, hours_worked, indexed, jobless, joblessness, jobs, labor, lagging, layoff, lfpr, maximum_employment, measures, nonsupervisory_workers, objectives, payroll_employment, per_hour, per_month, population_ratio, product, progress_toward, protected, pursuit, rate, reconcile, resolute, seeks_to_foster, statutory, switchers, thereby_promoting, tightness, tips, understated, underutilization, unemployment, unit, wages, whites, women, workweeks




#### Contraste con clustering automático (K-Means)

Las categorías anteriores las definí yo a partir de criterio económico. Para contrastarlas, aplico 
K-Means sobre los mismos términos: un algoritmo que agrupa sin intervención, solo por proximidad en 
el espacio de embeddings. Si los grupos automáticos coinciden con mis categorías temáticas, es 
evidencia de que los embeddings capturaron la estructura económica del corpus y no una organización 
arbitraria.

Uso K=6, la misma cantidad que mis categorías, para que la comparación sea directa. El clustering se 
aplica sobre los vectores originales de 300 dimensiones, no sobre la proyección 2D: t-SNE se usa solo 
para dibujar, y agrupar sobre una proyección ya distorsionada daría resultados menos confiables.

In [41]:
from sklearn.cluster import KMeans

K = 6

# Clustering sobre los vectores ORIGINALES de 300 dimensiones (no sobre la proyección 2D)
km = KMeans(n_clusters=K, random_state=42, n_init=10)
clusters = km.fit_predict(vectores_f)

fig = px.scatter(
    x=coords[:, 0], y=coords[:, 1], text=palabras_f,
    color=[f"Cluster {c}" for c in clusters],
    title=f"Embeddings FOMC en 2D — {K} grupos formados automáticamente (K-Means)"
)
fig.update_traces(textposition='top center', marker=dict(size=8))
fig.update_layout(height=850, width=1200)
fig.show()

# Composición de cada cluster para poder interpretarlos
print("Composición de los clusters:\n")
for c in range(K):
    miembros = [w for w, cl in zip(palabras_f, clusters) if cl == c]
    print(f"Cluster {c} ({len(miembros)} términos):")
    print("  " + ", ".join(miembros))
    print()

Composición de los clusters:

Cluster 0 (5 términos):
  price, stability, maximum_employment, goals, objectives

Cluster 1 (34 términos):
  economic, policy, monetary, financial, outlook, longer, risks, expectations, could, target_range, fomc, many, pressures, objective, stance, risk, action, accommodative, downside, significant, global, economies, balance_sheet, ongoing, effective, fiscal, observed, concerns, saw, easing, viewed, central, projections, situation

Cluster 2 (37 términos):
  market, reserve, securities, bank, open, markets, foreign, treasury, system, credit, banks, agency, operations, purchases, new_york, debt, holdings, domestic, transactions, loans, asset, mortgage_backed, currency, currencies, liquidity, government, international, issuance, change, commercial, shall, mbs, money, provide, small, funding, united_states

Cluster 3 (24 términos):
  pace, consumer, spending, business, sector, demand, sales, housing, investment, household, firms, expansion, manufacturing, s

![Clustering K-Means](img/kmeans.png)

#### Interpretación de los grupos

**Correspondencia entre categorías propias y clustering automático.** Cuatro de mis cinco categorías tienen correspondencia casi directa con los grupos que formó K-Means sin intervención:

- **Cluster 2** (37 términos) coincide con *Mercados financieros*: `treasury`, `securities`, `holdings`, `mortgage_backed`, `mbs`, `agency`, `debt`, `loans`, `credit`, `banks`, `operations`, `liquidity`, `funding`, `issuance`.
- **Cluster 3** (24) coincide con *Actividad económica*: `consumer`, `spending`, `business`, `sector`, `demand`, `housing`, `investment`, `manufacturing`, `capital`, `expenditures`, `firms`.
- **Cluster 1** (34) coincide con *Política monetaria*: `policy`, `monetary`, `stance`, `action`, `accommodative`, `easing`, `target_range`, `fomc`, `balance_sheet`, `risks`, `outlook`.
- **Cluster 5** (20) agrupa tasas y precios de activos: `rate`, `rates`, `funds`, `interest`, `yields`, `spreads`, `dollar`, `equity`, `mortgage`.

Que un algoritmo sin supervisión reproduzca agrupamientos equivalentes a los que definí por criterio económico indica que los embeddings capturaron la estructura temática real del corpus, y no una organización arbitraria.

**La excepción, y el hallazgo principal: el Cluster 4 fusiona inflación, empleo y actividad real.** Donde yo había separado *Precios e inflación* de *Mercado laboral*, el algoritmo agrupó `inflation`, `core`, `pce` junto con `unemployment`, `employment`, `labor`, `compensation`, y además con `growth`, `output`, `real_gdp`, `production` y `activity`.

Esta fusión era esperable. El marco analítico de la Fed en materia de inflación y empleo parte en buena medida de la curva de Phillips y sus desarrollos poskeynesianos, que postulan una relación entre desempleo e inflación y que han demostrado ser un instrumento robusto para la toma de decisiones de política económica. Que el cluster incorpore además el producto y la actividad real corresponde al trío estándar del marco macroeconómico: inflación, desempleo y producto no se analizan por separado sino como un sistema. En las actas del FOMC estas variables no se discuten de forma aislada, tanto porque inflación y empleo constituyen los dos términos del mandato dual como porque el marco teórico subyacente las vincula. El modelo, al medir contextos de coocurrencia, reprodujo esa relación analítica.

**El Cluster 0 refuerza esta lectura.** Con solo cinco términos —`price`, `stability`, `maximum_employment`, `goals`, `objectives`— constituye la enunciación literal del mandato dual ("price stability and maximum employment"). El algoritmo lo aisló como grupo propio, separándolo tanto del vocabulario analítico de inflación como del de empleo. Esto coincide con lo que había observado en el análisis de términos similares: `maximum_employment` pertenece al registro normativo-institucional, no al analítico.

**Grupo enfocado: el vecindario del mercado de trabajo.** La visualización específica del vocabulario laboral muestra cuatro dimensiones bien diferenciadas:

1. **Precio del trabajo:** `compensation`, `wage`, `wages`, `per_hour`, `hourly`, `average_hourly`, `cost_index`, `eci` (Employment Cost Index).
2. **Cantidad y utilización:** `unemployment`, `employment`, `payroll_employment`, `jobs`, `jobless`, `layoff`, `workweek`, `hours_worked`, `force_participation`, `population_ratio`, `underutilization`, `tightness`.
3. **Composición demográfica:** `african_americans`, `hispanics`, `whites`, agrupados y separados del resto.
4. **Marco normativo:** `maximum_employment`, `dual_mandate`, `statutory`, `objectives`, `goals`, `achieve_maximum`, `progress_toward`.

La separación entre precio y cantidad del trabajo reproduce la distinción analítica estándar en economía laboral. La presencia de un grupo demográfico diferenciado indica que el corpus trata el desempleo también en su dimensión distributiva y no solo agregada. Y la ubicación de `maximum_employment` en el extremo normativo, alejado de `unemployment`, confirma visualmente la decisión que tomé al elegir los términos de interés: son términos de registros distintos.

### Cierre del desafío

Entrené embeddings propios con Word2Vec sobre las actas y comunicados del FOMC (2000-2026), un corpus 
de 1,8 millones de palabras. Las conclusiones principales:

**Sobre el corpus y el preprocesamiento.** El análisis exploratorio previo fue determinante para las 
decisiones posteriores. Permitió descartar `argentina` como término de interés por baja frecuencia (27 
apariciones), detectar que las actas concentran el 95% del texto, e identificar que el vocabulario más 
frecuente está contaminado por fórmulas administrativas. El análisis de n-gramas justificó unir 
bigramas económicos en tokens únicos, decisión que requirió calibrar los umbrales: con valores 
permisivos el algoritmo pegaba frases enteras, y con valores exigentes se perdieron bigramas frecuentes 
pero poco exclusivos, como `labor_market`. Es un trade-off que asumí explícitamente.

**Sobre lo que capturan los embeddings.** El modelo aprendió el vocabulario técnico de la política 
monetaria con precisión: `inflation` se asocia a `headline`, `core`, `expectations` y `unmooring`; 
`unemployment` a `lfpr`, `epop` y `force_participation`. También capturó regularidades no obvias: 
`crisis` agrupa tres episodios distintos del período (la crisis financiera de 2008 vía `gfc`, la 
pandemia y Silicon Valley Bank en 2023), porque el corpus los describe con vocabulario similar.

**Sobre las limitaciones del método.** El caso de `easing` es ilustrativo: su vecino más cercano es 
`tightening`, su antónimo. Word2Vec mide contexto y no significado, de modo que palabras 
intercambiables sintácticamente resultan vectorialmente cercanas aunque signifiquen lo contrario. Del 
mismo modo, los términos "menos similares" no son antónimos sino vocabulario de contextos ajenos, por 
lo que aportan poco al análisis.

**Sobre la estructura del corpus.** El contraste entre mis categorías temáticas y el clustering 
automático mostró que cuatro de cinco categorías tienen correspondencia directa con los grupos que 
formó K-Means sin supervisión, lo que indica que los embeddings capturaron la estructura económica 
real del corpus. La excepción es reveladora: el algoritmo fusiona inflación y empleo en un solo grupo, 
algo esperable dado que el marco analítico de la Fed en esta materia parte de la curva de Phillips y 
sus desarrollos poskeynesianos, y que ambas variables constituyen los dos términos de su mandato dual. 
El discurso las trata como un objeto conjunto, y el modelo lo reprodujo.

**Sobre la distinción entre registros.** Un resultado transversal es que el corpus contiene al menos 
dos registros diferenciables: el analítico (variables, mediciones, indicadores) y el 
normativo-institucional (mandato, objetivos legales, formalidades de las actas). Los embeddings los 
separan con nitidez, como se ve en la ubicación de `maximum_employment` —término jurídico— alejado de 
`unemployment` —variable económica—. Esa distinción, que detecté al elegir los términos de interés, se 
confirmó luego en la visualización.

**Aplicabilidad.** La técnica es directamente trasladable al análisis de discursos de política 
económica en otros contextos, que es el uso que me interesa para mi investigación. El principal 
requisito es el volumen de corpus: con 1,8 millones de palabras los embeddings resultaron estables 
para términos frecuentes, pero los términos de baja frecuencia (menos de 50 apariciones) producen 
vectores poco confiables, como quedó demostrado al descartar `argentina`.